# Summarize

## Setup

In [ ]:
import os, sys
ROOT = os.path.abspath('..')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

from src import paths
print('project folder:', paths.PROJECT_ROOT)

from src.config import loadConfig
from src.summary import aggregate, aggregateClasses, collectRuns, pavpuTable, perSeedEffect, thresholdWarnings

In [ ]:
DATASETS = ['paris_lille', 'toronto3d', 'ahn3','vaihingen']

In [ ]:
runs, classes = collectRuns(DATASETS)
if runs.empty:
    raise SystemExit('no evaluated runs found. run notebook 03 first')
for w in thresholdWarnings(runs):
    print('WARNING:', w)
outRoot = paths.outputRoot(loadConfig(DATASETS[0]))
runs.to_csv(os.path.join(outRoot, 'pnpp_runs.csv'), index=False)
display(runs.round(4))

## Mean ± std over seeds

In [ ]:
agg = aggregate(runs)
agg.to_csv(os.path.join(outRoot, 'pnpp_summary.csv'))
display(agg)

## Uncertainty effect per seed (diff between uncertainty and baseline, same seed)

In [ ]:
eff = perSeedEffect(runs)
display(eff.round(4))
if not eff.empty:
    display(eff.groupby('dataset').agg(['mean', 'std', 'min', 'max']).round(4))

## PAvPU breakdown, mean over seeds
PAvPU = 0.5 when uncertainty is unrelated to errors

In [ ]:
pav = pavpuTable(runs)
pav.to_csv(os.path.join(outRoot, 'pnpp_pavpu.csv'))
display(pav)

## Per-class IoU, mean over seeds

In [ ]:
cls = aggregateClasses(classes)
cls.to_csv(os.path.join(outRoot, 'pnpp_per_class_iou.csv'))
for ds in DATASETS:
    if not cls.empty and ds in cls.index.get_level_values(0):
        print(f'\n{ds}')
        display(cls.loc[ds]['mean'])
print(f'\nCSV files written to {outRoot}')